# មេរៀន ០៩ - លំនាំរចនាបែប Metacognition


## ការតម្លើង

សៀវភៅកំណត់ត្រានេះបង្ហាញពីគំរូរចនាបថ Metacognition ដោយប្រើ Microsoft Agent Framework។

**ការទាមទារ មុនការប្រើប្រាស់៖**
- ការតម្លើង Azure OpenAI ត្រូវបានកំណត់តាមរយៈអថេរបរិស្ថាន
- បាន Authenticate ជាមួយ Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## តើ Metacognition ជាអ្វី?

Metacognition គឺជា **ការគិតអំពីការគិត**។ ក្នុងបរិបទនៃភ្នាកិ្ត AI វាមានន័យថាបង្កើតភ្នាកិ្តដែលអាច:

- **ពិចារណារូបខ្លួន** លើលទ្ធផលនិងដំណើរការត្រួតពិនិត្យរបស់ខ្លួន
- **រកឃើញកំហុស** និងស្ដារឡើងវិញដោយយ៉ាងរលូនជំនួសការបរាជ័យដោយស្ងាត់
- **ធ្វើការវាយតម្លៃ** ថាតើចម្លើយរបស់ពួកគេខ្ពស់គ្រប់គ្រាន់និងមានប្រយោជន៍ឬអត់
- **បញ្ចេញ** វិធីសាស្រ្តរបស់ពួកគេលើពេលដែលវិធីដំបូងមិនប្រើបាន (ឧ. ព្យាយាមវិធីបញ្ចូលជំនួយបម្រុង)

ភ្នាកិ្ត metacognitive មិនត្រឹមតែឆ្លើយសំណួរ — វាក៏ត្រួតពិនិត្យការធ្វើការ​របស់ខ្លួនផងហើយកែសំរួលភ្លាមៗផងដែរ។


## ឧបករណ៍មូលដ្ឋាន និង ជំនួយ

រាងមុខមេតាការច្នៃប្រឌិតទូទៅគឺ **យុទ្ធសាស្ត្រចុះបន្ទាប់**។ អ្នកតំណាងព្យាយាមប្រើឧបករណ៍មូលដ្ឋានជាលើកដំបូង; ប្រសិនបើវាខកច្រឡំ (ឧ. កំហុស 404) អ្នកតំណាងនឹងស្គាល់កំហុសនោះ ហើយយ៉ាងច្បាស់ជំនួសទៅឧបករណ៍ជំនួយ។

វាបញ្ជាក់ដូចជាប្រព័ន្ធនៅពិភពជាក់ស្តែងដែលសេវាកម្មមូលដ្ឋានអាចមិនមានឬអាចមិនមានប្រសិទ្ធភាព ក៏ដូចជាអ្នកតំណាងត្រូវតែប្រើប្រាស់អ្វីដែលលើកឡើងមុនពេលជ្រើសរើសផ្លូវជំនួសមួយ។

ខាងក្រោមយើងកំណត់ឧបករណ៍រកមើលเที่ยวบินពីរដូចខាងក្រោម៖
- **មូលដ្ឋាន** — គ្របដណ្តប់ទីក្រុងប៉ារីស ទូក្យូ និងបារុស្សាឡូណា
- **ជំនួយ** — គ្របដណ្តប់ទីក្រុងបឺរឡាំង ស៊ីដនី និងទីក្រុងញូវយ៉ក


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## អ្នកតំណាងផ្ទាល់ខ្លួនមានការពិចារណាផ្ទាល់ខ្លួន និងការស្ដារឡើងវិញបន្ទប់កំហុស

អ្នកតំណាងខាងក្រោមត្រូវបានបញ្ជូនឱ្យព្យាយាមប្រព័ន្ធហោះដាដំបូង ជ្រាបករណីបរាជ័យ ហើយបើក្សំព្យាយាមដល់ប្រព័ន្ធកម្លាំងបម្រុងមធ្យម។ បន្ទាប់ពីការឆ្លើយតបរៀងរាល់ការត្រួតពិនិត្យបន្ទាន់ តើវាបានឆ្លើយសំណួររបស់អ្នកប្រើប្រាស់បានពេញលេញឬអត់។


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## គំរូការវាយតម្លៃខ្លួនឯង

មុខងារផ្សេងទៀតនៃការយល់ដឹងពីខ្លូនឯងគឺ **ការវាយតម្លៃខ្លួនឯង**: អ្នកតំណាងផ្សេងម្នាក់ទេ (ឬអ្នកតំណាងដូចគ្នានៅដំណាក់កាលទីពីរ) បានពិនិត្យជាមុនលើចម្លើយសម្រាប់ភាពពេញលេញ, ភាពត្រឹមត្រូវ, និងភាពមានប្រយោជន៍។

ខាងក្រោមយើងបង្កើតអ្នកតំណាង `ResponseEvaluator` ដែលវាយតម្លៃចម្លើយរបស់ភ្នាក់ងារធ្វើដំណើរនៅលើបីវិមាត្រ។


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនរបៀបសាងសង់ **ភ្នាក់ងារមេតាចំណេះដឹង** ដោយប្រើ Microsoft Agent Framework ៖

- **ការត្រួតពិនិត្យខ្លួនឯង**៖ ភ្នាក់ងារដែលត្រួតពិនិត្យគំនិតរបស់ខ្លួនឯង និងសំដែងយ៉ាងច្បាស់ថាអ្វីបានកើតឡើង។
- **ការស្ដារឡើងវិញដោយមានជម្រើសជំនួស**៖ គំរូឧបករណ៍មួយដ៏សំខាន់ + ជំនួស ដែលភ្នាក់ងារស្វែងរកការខូចខាត (ឧទាហរណ៍ កំហុស 404) ហើយព្យាយាមប្រភពជំនួសដោយស្វ័យប្រវត្តិ។
- **ការវាយតម្លៃខ្លួនឯង**៖ ភ្នាក់ងារវាយតម្លៃផ្ទាល់ខ្លួនមួយដែលផ្ដល់ពិន្ទុចំពោះចម្លើយដោយគិតពីភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពជួយសម្រួល។

គំរូទាំងនេះធ្វើឲ្យភ្នាក់ងារមានភាពរឹងមាំច្បាស់លាស់ និងជឿទុកចិត្តបាន — គុណលក្ខណៈសំខាន់សម្រាប់ការប្រើប្រាស់នៅចំងាយផលិតកម្ម។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
